# **Model Comparison: TF-IDF + Logistic Regression vs Fine-Tuned Qwen vs RAG + Qwen**

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# 1. TF-IDF + Logistic Regression

### Load the test Data

In [2]:
import json

TEST_PATH = "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/test.jsonl"

test_data = []

with open(TEST_PATH, "r") as f:
    for line in f:
        test_data.append(json.loads(line))

print("Number of test emails:", len(test_data))
print("First test email:")
print(test_data[0])

Number of test emails: 5954
First test email:
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': 'Subject: re : credit risk technical article\n\nBody: ben ,\r\nlet me review it one more time from this angle . if you don \' t hear from me by\r\nwednesday ,\r\nit \' s ok to post it .\r\nvince\r\nbenjamin parsons\r\n01 / 04 / 2000 04 : 31 am\r\nto : vince j kaminski / hou / ect @ ect\r\ncc : ross prevatt / hou / ect @ ect , bijoya banerjea / lon / ect @ ect\r\nsubject : credit risk technical article\r\nvince ,\r\nbijoya is currently working on the development of the website for the new\r\ncredit trading initiative , and we thought it would be a good idea to put my\r\nrecent technical article " measuring credit risk " on it . are you okay with\r\nthis release , or does it contain any proprietary information we should edit\r\nout ?\r\nben', 'output': 'legitimate'}


### Check the test-set label distribution

In [3]:
from collections import Counter

label_counts = Counter(
    item["output"]
    for item in test_data
)

print("Test set label distribution:")
print(label_counts)

Test set label distribution:
Counter({'legitimate': 3151, 'phishing': 2803})


### Load the training data

In [4]:
TRAIN_PATH = "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/train.jsonl"

train_data = []

with open(TRAIN_PATH, "r") as f:
    for line in f:
        train_data.append(json.loads(line))

print("Number of training emails:", len(train_data))
print("First training email:")
print(train_data[0])

Number of training emails: 23813
First training email:
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': "Subject: barrister thambo\n\nBody: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s\r\nhowever the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?\r\nexpecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com\r\nregard ' s\r\nbarrister andrew thambo", 'output': 'phishing'}


### Prepare the text and labels

In [5]:
# Puts JSONL data into scikit-learn expects.

X_train = [item["input"] for item in train_data]
y_train = [item["output"] for item in train_data]

X_test = [item["input"] for item in test_data]
y_test = [item["output"] for item in test_data]

print("Training samples:", len(X_train))
print("Training labels:", len(y_train))

print("Test samples:", len(X_test))
print("Test labels:", len(y_test))

print("\nExample training text:")
print(X_train[0])

print("\nExample training label:")
print(y_train[0])

Training samples: 23813
Training labels: 23813
Test samples: 5954
Test labels: 5954

Example training text:
Subject: barrister thambo

Body: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s
however the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?
expecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com
regard ' s
barrister andrew thambo

Example training label:
phishing


### Create TF-IDF representation

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF training shape: (23813, 50000)
TF-IDF test shape: (5954, 50000)


### Train Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression

tfidf_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_classifier.fit(
    X_train_tfidf,
    y_train
)

print("TF-IDF classifier trained successfully!")

TF-IDF classifier trained successfully!


### Make predictions on the test set

In [8]:
tfidf_predictions = tfidf_classifier.predict(X_test_tfidf)

print("Number of predictions:", len(tfidf_predictions))
print("First 10 predictions:")
print(tfidf_predictions[:10])

Number of predictions: 5954
First 10 predictions:
['legitimate' 'phishing' 'phishing' 'phishing' 'legitimate' 'phishing'
 'legitimate' 'legitimate' 'legitimate' 'phishing']


### Calculate TF-IDF metrics

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

tfidf_accuracy = accuracy_score(
    y_test,
    tfidf_predictions
)

tfidf_precision = precision_score(
    y_test,
    tfidf_predictions,
    pos_label="phishing"
)

tfidf_recall = recall_score(
    y_test,
    tfidf_predictions,
    pos_label="phishing"
)

tfidf_f1 = f1_score(
    y_test,
    tfidf_predictions,
    pos_label="phishing"
)

print("TF-IDF Classification Results")
print("-----------------------------")
print(f"Accuracy : {tfidf_accuracy:.4f}")
print(f"Precision: {tfidf_precision:.4f}")
print(f"Recall   : {tfidf_recall:.4f}")
print(f"F1-score : {tfidf_f1:.4f}")

print("\nDetailed classification report:")
print(
    classification_report(
        y_test,
        tfidf_predictions
    )
)

TF-IDF Classification Results
-----------------------------
Accuracy : 0.9871
Precision: 0.9762
Recall   : 0.9968
F1-score : 0.9864

Detailed classification report:
              precision    recall  f1-score   support

  legitimate       1.00      0.98      0.99      3151
    phishing       0.98      1.00      0.99      2803

    accuracy                           0.99      5954
   macro avg       0.99      0.99      0.99      5954
weighted avg       0.99      0.99      0.99      5954



In [10]:
print("TF-IDF Classification Results (%)")
print("----------------------------------")
print(f"Accuracy : {tfidf_accuracy * 100:.2f}%")
print(f"Precision: {tfidf_precision * 100:.2f}%")
print(f"Recall   : {tfidf_recall * 100:.2f}%")
print(f"F1-score : {tfidf_f1 * 100:.2f}%")

TF-IDF Classification Results (%)
----------------------------------
Accuracy : 98.71%
Precision: 97.62%
Recall   : 99.68%
F1-score : 98.64%


# 2. Fine-Tuned Qwen

### Check for existing Qwen results

In [11]:
print("Checking for previous Qwen evaluation results...")

qwen_variables = [
    "qwen_predictions",
    "qwen_accuracy",
    "qwen_precision",
    "qwen_recall",
    "qwen_f1"
]

for variable in qwen_variables:
    print(f"{variable}: {'available' if variable in globals() else 'not available'}")

Checking for previous Qwen evaluation results...
qwen_predictions: not available
qwen_accuracy: not available
qwen_precision: not available
qwen_recall: not available
qwen_f1: not available


### Find Qwen Notebook

In [12]:
import os

PROJECT_PATH = "/content/drive/MyDrive/PhishDetectAI"

for root, dirs, files in os.walk(PROJECT_PATH):
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))

/content/drive/MyDrive/PhishDetectAI/notebooks/exploration_1.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/exploration_2.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/preparation.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/qwen_finetuning.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/RAG.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/streamlit_ui.ipynb
/content/drive/MyDrive/PhishDetectAI/notebooks/evaluation.ipynb


### Search the Qwen notebook for evaluation code

In [13]:
import json

QWEN_NOTEBOOK = "/content/drive/MyDrive/PhishDetectAI/notebooks/qwen_finetuning.ipynb"

with open(QWEN_NOTEBOOK, "r") as f:
    qwen_nb = json.load(f)

print("Searching Qwen notebook...\n")

for i, cell in enumerate(qwen_nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(keyword in source.lower() for keyword in [
        "5954",
        "prediction",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "classification_report"
    ]):
        print(f"--- Cell {i} ---")
        print(source[:3000])
        print()

Searching Qwen notebook...

--- Cell 19 ---
# Baseline Inference (Before Fine-Tuning)

# Take one example from the test set
sample = qwen_dataset["test"][0]

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": sample["messages"][1]["content"]
    }
]

# Format the prompt using Qwen's chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate prediction
output = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract only the generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("EMAIL:")
print(sample["messages"][1]["content"])

print("\nEXPECTED LABEL:")
print(sample["messages"][2]

### Run the saved Qwen metrics

In [17]:
# Find and display the saved output of the Qwen evaluation cells

for i in [37, 38]:
    cell = qwen_nb["cells"][i]

    print(f"\n===== CELL {i} =====")

    outputs = cell.get("outputs", [])

    if not outputs:
        print("No saved output in this cell.")
        continue

    for output in outputs:
        if "text" in output:
            print("".join(output["text"]))

        if "data" in output:
            if "text/plain" in output["data"]:
                print("".join(output["data"]["text/plain"]))


===== CELL 37 =====
==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Evaluating model:   0%|          | 0/2977 [00:00<?, ?it/s]
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/e

In [18]:
qwen_accuracy = 99.20
qwen_precision = 99.65
qwen_recall = 98.82
qwen_f1 = 99.23

print("Fine-Tuned Qwen Results")
print("-----------------------")
print(f"Test samples: 2977")
print(f"Valid predictions: 2735")
print(f"Invalid predictions: 242")
print(f"Accuracy:  {qwen_accuracy:.2f}%")
print(f"Precision: {qwen_precision:.2f}%")
print(f"Recall:    {qwen_recall:.2f}%")
print(f"F1-score:  {qwen_f1:.2f}%")

Fine-Tuned Qwen Results
-----------------------
Test samples: 2977
Valid predictions: 2735
Invalid predictions: 242
Accuracy:  99.20%
Precision: 99.65%
Recall:    98.82%
F1-score:  99.23%


## As qwen is doing 2977 so we do same for Logistic regression

### Find the exact 2,977 test split

In [19]:
for i, cell in enumerate(qwen_nb["cells"]):
    source = "".join(cell.get("source", []))

    if "qwen_dataset" in source and (
        "train_test_split" in source
        or "DatasetDict" in source
        or "validation" in source
        or "test" in source
    ):
        print(f"--- Cell {i} ---")
        print(source[:5000])
        print()

--- Cell 19 ---
# Baseline Inference (Before Fine-Tuning)

# Take one example from the test set
sample = qwen_dataset["test"][0]

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": sample["messages"][1]["content"]
    }
]

# Format the prompt using Qwen's chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate prediction
output = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract only the generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("EMAIL:")
print(sample["messages"][1]["content"])

print("\nEXPECTED LABEL:")
print(sample["messages"][2]["content"])

print("\nSTART

### Find the dataset split code

In [20]:
for i, cell in enumerate(qwen_nb["cells"]):
    source = "".join(cell.get("source", []))

    if "qwen_dataset =" in source:
        print(f"--- Cell {i} ---")
        print(source)

--- Cell 13 ---
qwen_dataset = final_dataset.map(
    convert_to_qwen_messages
)


### Reproduce exact Qwen test set

In [21]:
from datasets import Dataset

# Convert the 5954 prepared test samples into a Hugging Face Dataset
prepared_test_dataset = Dataset.from_list(test_data)

# Reproduce the exact split used for Qwen
test_split = prepared_test_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

# The same test portion used by Qwen
qwen_test_data = test_split["test"]

print("Validation samples:", len(test_split["train"]))
print("Qwen test samples:", len(qwen_test_data))

print("\nFirst Qwen test sample:")
print(qwen_test_data[0])

Validation samples: 2977
Qwen test samples: 2977

First Qwen test sample:
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': 'Subject: abb contract\n\nBody: dave ,\r\ni am still diving into the depths of tex - mex , but cannot see a reason not to proceed with signing the abb amendment as described in salsa ( 4 ) , as this amendment does not cost us anything up front and extends our build option to the end of the year .\r\nthere are a number of risks associated with the presidential permit process , the possible imposition of an open season by ferc , relationship with cfe , bpub infrastructure enhancements and service charge structure , etc . which we should discuss prior to waiving our conditions precedent under the deacero contract on july 31 .\r\nnote that if we sign the abb amendment and waive the conditions precedent along with deacero on july 31 , we will want to immediately issue a pre - notice of the notice to proceed in order to ensure completio

## Prepare the exact Qwen test data

In [22]:
# Convert the exact Qwen test split into lists

qwen_X_test = [item["input"] for item in qwen_test_data]
qwen_y_test = [item["output"] for item in qwen_test_data]

print("Qwen test texts:", len(qwen_X_test))
print("Qwen test labels:", len(qwen_y_test))

Qwen test texts: 2977
Qwen test labels: 2977


### Load the training data again

In [23]:
TRAIN_PATH = "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/train.jsonl"

train_data = []

with open(TRAIN_PATH, "r") as f:
    for line in f:
        train_data.append(json.loads(line))

X_train = [item["input"] for item in train_data]
y_train = [item["output"] for item in train_data]

print("Training samples:", len(X_train))

Training samples: 23813


### Create TF-IDF representation

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

print("TF-IDF training shape:", X_train_tfidf.shape)

TF-IDF training shape: (23813, 50000)


### Train the Logistic Regression classifier

In [25]:
from sklearn.linear_model import LogisticRegression

tfidf_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_classifier.fit(
    X_train_tfidf,
    y_train
)

print("TF-IDF classifier trained successfully!")

TF-IDF classifier trained successfully!


### create TF-IDF test vectors

In [26]:
qwen_X_test_tfidf = tfidf_vectorizer.transform(qwen_X_test)

print("TF-IDF test shape:", qwen_X_test_tfidf.shape)

TF-IDF test shape: (2977, 50000)


### Predictions

In [27]:
tfidf_qwen_test_predictions = tfidf_classifier.predict(
    qwen_X_test_tfidf
)

print("Number of predictions:", len(tfidf_qwen_test_predictions))
print("First 10 predictions:")
print(tfidf_qwen_test_predictions[:10])

Number of predictions: 2977
First 10 predictions:
['legitimate' 'phishing' 'phishing' 'legitimate' 'phishing' 'phishing'
 'phishing' 'phishing' 'phishing' 'phishing']


### Fair TF-IDF metrics

In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

tfidf_qwen_accuracy = accuracy_score(
    qwen_y_test,
    tfidf_qwen_test_predictions
)

tfidf_qwen_precision = precision_score(
    qwen_y_test,
    tfidf_qwen_test_predictions,
    pos_label="phishing"
)

tfidf_qwen_recall = recall_score(
    qwen_y_test,
    tfidf_qwen_test_predictions,
    pos_label="phishing"
)

tfidf_qwen_f1 = f1_score(
    qwen_y_test,
    tfidf_qwen_test_predictions,
    pos_label="phishing"
)

print("TF-IDF Results — Same Test Set as Qwen")
print("---------------------------------------")
print(f"Test samples: {len(qwen_y_test)}")
print(f"Accuracy : {tfidf_qwen_accuracy * 100:.2f}%")
print(f"Precision: {tfidf_qwen_precision * 100:.2f}%")
print(f"Recall   : {tfidf_qwen_recall * 100:.2f}%")
print(f"F1-score : {tfidf_qwen_f1 * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    qwen_y_test,
    tfidf_qwen_test_predictions
))

TF-IDF Results — Same Test Set as Qwen
---------------------------------------
Test samples: 2977
Accuracy : 98.62%
Precision: 97.50%
Recall   : 99.65%
F1-score : 98.56%

Classification Report:
              precision    recall  f1-score   support

  legitimate       1.00      0.98      0.99      1566
    phishing       0.98      1.00      0.99      1411

    accuracy                           0.99      2977
   macro avg       0.99      0.99      0.99      2977
weighted avg       0.99      0.99      0.99      2977



# 3. RAG + Qwen

### Search for RAG notebook for evaluation

In [29]:
import json

QWEN_NOTEBOOK = "/content/drive/MyDrive/PhishDetectAI/notebooks/RAG.ipynb"

with open(QWEN_NOTEBOOK, "r") as f:
    rag_nb = json.load(f)

print("Searching Qwen notebook...\n")

for i, cell in enumerate(rag_nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(keyword in source.lower() for keyword in [
        "5954",
        "prediction",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "classification_report"
    ]):
        print(f"--- Cell {i} ---")
        print(source[:3000])
        print()

Searching Qwen notebook...

--- Cell 35 ---
import numpy as np

# Evaluate on the first 1000 test emails
EVAL_SIZE = 1000

top1_correct = 0
top3_correct = 0

for i in range(EVAL_SIZE):

    # Create embedding for the test email
    query_embedding = embedding_model.encode(
        test_documents[i],
        show_progress_bar=False
    ).tolist()

    # Retrieve top 3 similar emails
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["metadatas"]
    )

    retrieved_labels = [
        metadata["label"]
        for metadata in results["metadatas"][0]
    ]

    actual_label = test_labels[i]

    # Top-1
    if retrieved_labels[0] == actual_label:
        top1_correct += 1

    # Top-3
    if actual_label in retrieved_labels:
        top3_correct += 1

top1_recall = top1_correct / EVAL_SIZE
top3_recall = top3_correct / EVAL_SIZE

print("RAG Retrieval Evaluation")
print("=" * 40)
print(f"Evaluation emails: {EVAL_SIZE}")
print(f

In [31]:
print("Number of cells in RAG notebook:", len(rag_nb["cells"]))

Number of cells in RAG notebook: 71


In [32]:
for i in [70]:
    cell = rag_nb["cells"][i]

    print(f"\n===== CELL {i} =====")

    outputs = cell.get("outputs", [])

    if not outputs:
        print("No saved output in this cell.")
        continue

    for output in outputs:
        if "text" in output:
            print("".join(output["text"]))

        if "data" in output:
            if "text/plain" in output["data"]:
                print("".join(output["data"]["text/plain"]))


===== CELL 70 =====
RAG + Qwen FINAL EVALUATION
Total emails       : 2977
Valid predictions  : 2922
Invalid predictions: 55

Accuracy : 0.9802 (98.02%)
Precision: 0.9715 (97.15%)
Recall   : 0.9870 (98.70%)
F1 Score : 0.9792 (97.92%)

Confusion Matrix
                 Predicted
                 Legitimate  Phishing
Actual Legitimate    1501          40
Actual Phishing        18        1363



## Model Comparison on 2977 Emails

Here's a summary of the performance for each model on the 2977 test emails:

| Metric      | TF-IDF + Logistic Regression | Fine-Tuned Qwen | RAG + Qwen |
| :---------- | :--------------------------- | :-------------- | :--------- |
| **Accuracy**| 98.62%                       | 99.20%          | 98.02%     |
| **Precision**| 97.50%                       | 99.65%          | 97.15%     |
| **Recall**  | 99.65%                       | 98.82%          | 98.70%     |
| **F1-Score**| 98.56%                       | 99.23%          | 97.92%     |
| **Invalid Predictions** | 0                            | 242             | 55         |